# 02 — Embeddings and Semantic Search

## Why this notebook exists

Notebook 01 left us with a precise problem: we need to send the model only the *relevant* documents, but to do that we have to *measure* relevance — and we hand-picked the right document ourselves, which is exactly the step that needs automating. Keyword matching is a weak proxy: a question about "how long the robot runs" should match a document that says "battery life," even though they share no words.

This notebook introduces the tool that solves it: **embeddings**. An embedding turns a piece of text into a vector of numbers, positioned so that text with similar *meaning* lands close together — even when the words differ. We'll look at what an embedding actually is, measure closeness with **cosine similarity** (computed by hand in NumPy, so nothing is hidden), watch semantic similarity beat keyword overlap, and then build a tiny search that ranks our Halcyon documents by relevance to a question. That ranking is the first real retrieval step — the automated version of notebook 01's hand-picking.

Requires an `OPENAI_API_KEY` (for the embeddings API). No vector database yet — just NumPy, so you can see exactly how it works.

## What you'll learn

- What an **embedding** is: a fixed-length vector of floats representing a piece of text, and how to get one from `text-embedding-3-small`.
- How to measure similarity between two embeddings with **cosine similarity**, written out in NumPy.
- Why **semantic** similarity beats **keyword** matching — a question and its answer can share almost no words yet still be close in embedding space.
- How to build a **brute-force top-k semantic search**: embed a corpus, embed a query, rank by cosine, return the best matches.
- Why embedding *whole documents* is only a starting point — and why notebook 03 needs to chunk them.

## 1. Setup

This notebook calls the OpenAI **embeddings** API and does all the math in NumPy. We load `OPENAI_API_KEY` from the environment (and from a local `.env` file if `python-dotenv` is installed — real environment variables always win), then define three small helpers:

- `embed(text)` — embed a single string into a NumPy vector.
- `embed_many(texts)` — embed a list of strings in one API call (cheaper and faster than looping).
- `cosine(a, b)` — cosine similarity between two vectors, a number in `[-1, 1]` where higher means more similar.

In [ ]:
import os

# Optional: load a local .env if python-dotenv is installed. Real env vars win.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ── Key guard ──────────────────────────────────────────────────────────────
if not os.environ.get("OPENAI_API_KEY"):
    print("=" * 60)
    print("OPENAI_API_KEY is not set.")
    print("=" * 60)
    print()
    print("This notebook calls the OpenAI embeddings API.")
    print()
    print("Set it and restart the kernel:")
    print("  export OPENAI_API_KEY=sk-...")
    raise SystemExit("Set OPENAI_API_KEY and restart the kernel to continue.")

print("OPENAI_API_KEY set ✓")

In [ ]:
import numpy as np
from openai import OpenAI

EMBED_MODEL = "text-embedding-3-small"

openai_client = OpenAI()  # reads OPENAI_API_KEY from the environment


def embed(text: str) -> np.ndarray:
    """Embed a single string into a NumPy vector."""
    resp = openai_client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(resp.data[0].embedding, dtype=np.float32)


def embed_many(texts: list) -> np.ndarray:
    """Embed a list of strings in ONE API call.

    Returns a matrix of shape (len(texts), embedding_dim); row i is the
    embedding of texts[i]. The API preserves input order.
    """
    resp = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data], dtype=np.float32)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors: the cosine of the angle between
    them. 1.0 = identical direction, 0.0 = unrelated (orthogonal), -1.0 = opposite.
    """
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("Setup OK")
print(f"Embedding model: {EMBED_MODEL}")